# Text incident classifier

Trains the text modality: short messages (helpline transcripts, social posts)
flagged as describing an emergency or not. A positive prediction feeds fusion as
corroborating evidence for whatever video/audio event is active — it does not
escalate on its own.

**Architecture:** mean-pooled `EmbeddingBag` + MLP head, in pure PyTorch. No
transformers, no sklearn — trains in well under a minute and the whole
checkpoint is a few MB. The vocabulary ships *inside* the checkpoint, so the app
needs no side files.

## Dataset

**Binary — `NORMAL` vs `INCIDENT`.** Do not try to split incidents into
ACCIDENT vs VIOLENCE by keyword: public text datasets are not labelled that
finely and the guesses are noise. The pipeline already treats a positive text
prediction as support for *both* event types (same as the audio model).

| Source (Kaggle) | Notes |
|---|---|
| **`vstepanenko/disaster-tweets`** | ~11k tweets, `text` + `target` columns. What this notebook expects. |
| **HumAID** (~77k crisis tweets, 10 categories) | Larger, more serious; collapse its categories to a binary flag |
| Your own helpline transcripts | Best domain fit if you can get them |

Add the dataset as an Input, then check the path:

```python
!ls /kaggle/input
```

**Scope honestly in the report:** disaster tweets are about earthquakes, floods,
and fires — not car crashes or fights. This demonstrates the *mechanism* (text
evidence entering fusion), not a model tuned to the system's specific incident
types. Live X/Twitter API access is paid; train on a static file and demo
through `POST /api/text`.

In [ ]:
import os, re, json, random, math
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cpu")   # tiny model; CPU is faster end-to-end
print("device:", DEVICE)

## 1. Tokenizer

**Copied verbatim from `src/cctv_ai/inference/text/text_classifier.py`.**
Train/inference tokenisation must match exactly.

In [ ]:
MAX_TOKENS, PAD, UNK = 64, 0, 1
_TOKEN_RE = re.compile(r"[a-z0-9']+")

def tokenize(text):
    text = (text or "").lower()
    text = re.sub(r"https?://\S+|www\.\S+", " <url> ", text)
    text = re.sub(r"@\w+", " <user> ", text)
    text = re.sub(r"#(\w+)", r" \1 ", text)
    return _TOKEN_RE.findall(text)[:MAX_TOKENS]

def encode(text, vocab):
    ids = [vocab.get(t, UNK) for t in tokenize(text)]
    return ids or [UNK]

## 2. Load and label

In [ ]:
LABEL_MAP = {"NORMAL": 0, "INCIDENT": 1}
# Point this at the file you added. For vstepanenko/disaster-tweets it is
# usually /kaggle/input/disaster-tweets/tweets.csv — check with `!ls /kaggle/input/*`.
CSV_PATH  = "/kaggle/input/disaster-tweets/tweets.csv"
TEXT_COL  = "text"
LABEL_COL = "target"       # 1 = disaster/incident, 0 = not

df = pd.read_csv(CSV_PATH)
df = df[[TEXT_COL, LABEL_COL]].dropna()
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df[df[LABEL_COL].isin([0, 1])]
print(df.shape, "|", df[LABEL_COL].value_counts().to_dict())

df["_label"] = df[LABEL_COL]

# Eyeball a few of each class — the labels are only as good as the source.
inv = {v: k for k, v in LABEL_MAP.items()}
for v in (0, 1):
    print(f"\n--- {inv[v]} samples ---")
    for t in df[df["_label"] == v][TEXT_COL].head(3):
        print("   ", str(t)[:110])

## 3. Vocabulary (built on train only — never on val)

In [ ]:
rows = list(zip(df[TEXT_COL].astype(str), df["_label"].astype(int)))
random.shuffle(rows)
split = int(0.85*len(rows))
train_rows, val_rows = rows[:split], rows[split:]

MIN_FREQ = 2
counter = Counter(t for txt,_ in train_rows for t in tokenize(txt))
vocab = {"<pad>": PAD, "<unk>": UNK}
for tok, c in counter.most_common():
    if c >= MIN_FREQ: vocab[tok] = len(vocab)
print("vocab size:", len(vocab), "| train:", len(train_rows), "val:", len(val_rows))

class TextDS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        txt, y = self.rows[i]
        return torch.tensor(encode(txt, vocab), dtype=torch.long), y

def collate(batch):
    ids = torch.cat([b[0] for b in batch])
    offsets = torch.tensor([0]+[len(b[0]) for b in batch[:-1]]).cumsum(0)
    return ids, offsets, torch.tensor([b[1] for b in batch], dtype=torch.long)

train_dl = DataLoader(TextDS(train_rows), batch_size=64, shuffle=True,  collate_fn=collate)
val_dl   = DataLoader(TextDS(val_rows),   batch_size=64, shuffle=False, collate_fn=collate)

## 4. Model

In [ ]:
EMBED_DIM = 128

class TextBagClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim=EMBED_DIM):
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode="mean", padding_idx=PAD)
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Sequential(nn.Linear(embed_dim,64), nn.ReLU(), nn.Linear(64,num_classes))
    def forward(self, ids, offsets):
        return self.head(self.dropout(self.embedding(ids, offsets)))

model = TextBagClassifier(len(vocab), len(LABEL_MAP)).to(DEVICE)

n = np.array([sum(1 for _,y in train_rows if y==v) for v in LABEL_MAP.values()], dtype=np.float64)
weights = torch.tensor(n.sum()/(len(n)*np.maximum(n,1)), dtype=torch.float32, device=DEVICE)
print("class weights:", weights.tolist())

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

## 5. Train

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 15
best = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train(); tot = cor = 0; ls = 0.0
    bar = tqdm(train_dl, desc=f"epoch {epoch:2d}/{EPOCHS}", leave=False)
    for ids, offsets, y in bar:
        ids, offsets, y = ids.to(DEVICE), offsets.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(ids, offsets); loss = criterion(out, y)
        loss.backward(); optimizer.step()
        ls += loss.item() * y.size(0); tot += y.size(0)
        cor += (out.argmax(1) == y).sum().item()
        bar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{cor/max(tot,1):.3f}")

    model.eval(); vt = vc = 0
    per = {v: [0, 0] for v in LABEL_MAP.values()}
    tp = fp = fn = 0
    with torch.inference_mode():
        for ids, offsets, y in tqdm(val_dl, desc="  val", leave=False):
            ids, offsets, y = ids.to(DEVICE), offsets.to(DEVICE), y.to(DEVICE)
            pred = model(ids, offsets).argmax(1)
            vc += (pred == y).sum().item(); vt += y.size(0)
            for v in LABEL_MAP.values():
                m = y == v
                per[v][0] += (pred[m] == v).sum().item(); per[v][1] += int(m.sum())
            tp += ((pred == 1) & (y == 1)).sum().item()
            fp += ((pred == 1) & (y == 0)).sum().item()
            fn += ((pred == 0) & (y == 1)).sum().item()

    va = vc / max(vt, 1)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    detail = "  ".join(f"{k} {per[v][0]/max(per[v][1],1):.2f}" for k, v in LABEL_MAP.items())
    print(f"epoch {epoch:2d}  loss {ls/max(tot,1):.4f}  train {cor/max(tot,1):.3f}  "
          f"val {va:.3f}  | {detail}  | incident P {prec:.3f} R {rec:.3f} F1 {f1:.3f}")

    # Save on F1, not accuracy: with class imbalance, accuracy rewards a model
    # that just predicts NORMAL.
    if f1 > best:
        best = f1
        torch.save({
            "model_name": "text",
            "arch": "embeddingbag_mean",
            "label_map": LABEL_MAP,
            "vocab": vocab,
            "embed_dim": EMBED_DIM,
            "max_tokens": MAX_TOKENS,
            "val_acc": va,
            "val_f1": f1,
            "state_dict": model.state_dict(),
        }, "text_best.pt")
        print(f"   saved text_best.pt  (F1 {f1:.3f}, P {prec:.3f}, R {rec:.3f})")

## 6. Sanity check before shipping

In [ ]:
model.eval()
probe = [
    "massive car crash on the highway three cars involved ambulance on scene",
    "huge fight broke out outside the stadium people are being attacked",
    "wildfire spreading fast near the town people evacuating now",
    "beautiful sunset today the weather is lovely",
    "just finished my homework going to sleep",
    "zzz qqq unknown gibberish words here",
]
inv = {v: k for k, v in LABEL_MAP.items()}
with torch.inference_mode():
    for t in probe:
        ids = torch.tensor(encode(t, vocab), dtype=torch.long)
        p = torch.softmax(model(ids, torch.tensor([0])), dim=1)[0]
        k = int(p.argmax())
        toks = tokenize(t)
        cov = sum(1 for tok in toks if tok in vocab) / max(len(toks), 1)
        print(f"{inv[k]:9s} {p[k]:.3f}  cov={cov:.2f}  {t[:60]}")

## Install

1. Download `text_best.pt` from `/kaggle/working`
2. Copy to `models/text_best.pt`
3. Set in `.env`:

```env
TEXT_MODEL_WEIGHTS_PATH=models/text_best.pt
```

4. Restart, then:

```bash
curl -X POST localhost:8000/api/text \
  -H 'Content-Type: application/json' \
  -d '{"text":"major road crash on MG road, several people injured"}'
```

The response includes `vocab_coverage` — the fraction of the message the model
recognised. The pipeline ignores predictions below `TEXT_MIN_VOCAB_COVERAGE`
(default 0.5), so an out-of-vocabulary message cannot become false evidence.